# 03. Fuel Efficiency Analysis — analiza efektywności paliwowej

**Projekt:** PRC Data Challenge 2025  
**Wejście:** `pk_big_data.curated.fuel_segments_train` (wyłącznie zbiór treningowy).  
**Cel:** porównanie *obserwowanego tempa zużycia paliwa* między typami samolotów, z uwzględnieniem czasu przedziału, długości trasy i jakości danych.

**Interpretacja:** `fuel_kg / dur_s × 3600` daje kg paliwa na godzinę **zaobserwowanych odcinków**, a nie efektywność na pasażerokilometr ani na rzeczywiście przeleciany kilometr. `route_km` jest odległością między lotniskami dla całego lotu, a nie długością analizowanego odcinka. Dane nie identyfikują konkretnej linii lotniczej ani rzeczywistego wieku egzemplarza samolotu.

## 0. Punkt wyjścia — ustalenia z notebooka 02 (EDA)

Wyniki dla zbioru treningowego: **131 530 przedziałów**, **75 kolumn**; brak NULL w `fuel_kg` i brak ujemnego zużycia paliwa; **mediana 200 kg**, średnia **557,71 kg**; **silna prawostronna asymetria** (maksimum 32 205,03 kg). `dur_s` ma korelację z `fuel_kg` **r = 0,789**, a `route_km` **r = 0,516**. Najkrótszy odcinek ma **1,61 s** mimo deklarowanego przez organizatorów progu 5 minut. Braki `gs_mean` i `vr_mean`: około **35,08%**, `mach_mean`: **35,97%**, `tas_mean`: **66,20%**, `cas_mean`: **83,77%**. `mach_std`, `tas_std` i `cas_std` są w całości brakujące.

**Konsekwencja:** nie porównujemy typów na podstawie samej średniej `fuel_kg` na odcinek. Przed obliczeniem tempa sprawdzamy krótkie odcinki, brakujące pomiary i obserwacje skrajne. Żadnych źródłowych wierszy nie usuwamy z tabeli Delta — ewentualne filtry dotyczą wyłącznie roboczych DataFrame'ów.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

CATALOG = "pk_big_data"
SCHEMA = "curated"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.fuel_segments_train"

train = spark.table(SOURCE_TABLE)
aircraft_col = "flight_aircraft_type" if "flight_aircraft_type" in train.columns else "aircraft_type"
required = {"flight_id", "fuel_kg", "dur_s", "route_km", aircraft_col}
missing = sorted(required - set(train.columns))
assert not missing, f"Brakuje kolumn: {missing}"

print(f"Dane: {SOURCE_TABLE}")
print("Liczba wierszy:", train.count())
print("Kolumna typu samolotu:", aircraft_col)
print("Liczba kolumn:", len(train.columns))

Dane: pk_big_data.curated.fuel_segments_train
Liczba wierszy: 131530
Kolumna typu samolotu: flight_aircraft_type
Liczba kolumn: 75


## 1. Audyt długości przedziałów i jakości pomiarów

Kontrolujemy przedziały krótsze niż 5 minut, czas niepoprawny, niedostępność punktów trajektorii i skrajnie wysokie tempa zużycia. **Nie zakładamy, że wszystkie krótkie odcinki są błędne** — najpierw liczymy ich skalę i pokazujemy wyniki z filtrem oraz bez filtra.

In [0]:
audit = train.agg(
    F.count("*").alias("n_all"),
    F.sum(F.when(F.col("dur_s").isNull() | (F.col("dur_s") <= 0), 1).otherwise(0)).alias("n_bad_duration"),
    F.sum(F.when((F.col("dur_s") > 0) & (F.col("dur_s") < 300), 1).otherwise(0)).alias("n_under_5min"),
    F.sum(F.when(F.col("dur_s") > 3600, 1).otherwise(0)).alias("n_over_60min"),
    F.sum(F.when(F.col("fuel_kg").isNull() | (F.col("fuel_kg") < 0), 1).otherwise(0)).alias("n_invalid_fuel"),
    F.sum(F.when(F.col(aircraft_col).isNull(), 1).otherwise(0)).alias("n_missing_type"),
    F.sum(F.when(F.col("route_km").isNull() | (F.col("route_km") < 0), 1).otherwise(0)).alias("n_missing_route"),
    F.sum(F.when(F.col("n_points") == 0, 1).otherwise(0)).alias("n_zero_trajectory")
       if "n_points" in train.columns else F.lit(None).cast("long").alias("n_zero_trajectory")
)
display(audit)

print("Przykładowe najkrótsze przedziały:")
display(train.select("flight_id", aircraft_col, "dur_s", "fuel_kg", "route_km")
        .orderBy("dur_s").limit(15))

n_all,n_bad_duration,n_under_5min,n_over_60min,n_invalid_fuel,n_missing_type,n_missing_route,n_zero_trajectory
131530,0,48856,0,0,0,0,3201


Przykładowe najkrótsze przedziały:


flight_id,flight_aircraft_type,dur_s,fuel_kg,route_km
prc779253614,A320,1.61,0.907184,3234.762957294575
prc779253614,A320,1.663,0.453592,3234.762957294575
prc772369375,A320,1.701,0.453592,850.2864002263814
prc783550373,A321,1.759,0.453592,1065.4393477586318
prc779253614,A320,1.775,1.814368,3234.762957294575
prc779253614,A320,1.852,1.814368,3234.762957294575
prc779253614,A320,1.868,1.360776,3234.762957294575
prc779253614,A320,1.873,0.907184,3234.762957294575
prc779253614,A320,2.214,1.360776,3234.762957294575
prc779689658,A320,2.478,1.360776,1712.7325504210298


## 2. Bezpieczne tempo spalania i warianty analizy

Podstawowa miara: `rate_kg_h = fuel_kg / dur_s × 3600`. Stosujemy ją tylko dla dodatniego czasu i nieujemnego paliwa. Do głównego porównania wybieramy przedziały **300–3600 s**, zgodne z deklarowaną metodologią konkursu. Równolegle zostawiamy wariant kontrolny obejmujący wszystkie dodatnie długości. To **analiza wrażliwości**, a nie cicha korekta danych.

Dla ograniczenia wpływu skrajnych odcinków pokazujemy zarówno średnie tempo ważone czasem (`sum(fuel_kg) / sum(dur_s)`), jak i medianę indywidualnych temp. Średnie ważone czasem nie powinno się interpretować jako miernika sprawności silnika.

In [0]:
base = (train
    .filter(F.col("fuel_kg").isNotNull() & (F.col("fuel_kg") >= 0))
    .filter(F.col("dur_s").isNotNull() & (F.col("dur_s") > 0))
    .filter(F.col(aircraft_col).isNotNull())
    .withColumn("rate_kg_h", F.col("fuel_kg") / F.col("dur_s") * F.lit(3600.0))
    .withColumn("co2_rate_kg_h", F.col("rate_kg_h") * F.lit(3.16))
    .withColumn("valid_duration_5_60", F.col("dur_s").between(300, 3600))
)

segments = base.filter(F.col("valid_duration_5_60"))
print("Wszystkie dodatnie przedziały:", base.count())
print("Przedziały 5–60 min:", segments.count())

for label, df in [("Wszystkie >0 s", base), ("Wyłącznie 5–60 min", segments)]:
    display(df.agg(
        F.lit(label).alias("wariant"),
        F.count("*").alias("n_segments"),
        F.round(F.sum("fuel_kg") / F.sum("dur_s") * 3600, 2).alias("rate_weighted_kg_h"),
        F.round(F.expr("percentile_approx(rate_kg_h, 0.5)"), 2).alias("median_rate_kg_h"),
        F.round(F.expr("percentile_approx(rate_kg_h, 0.99)"), 2).alias("p99_rate_kg_h")
    ))

Wszystkie dodatnie przedziały: 131530
Przedziały 5–60 min: 82674


wariant,n_segments,rate_weighted_kg_h,median_rate_kg_h,p99_rate_kg_h
Wszystkie >0 s,131530,3774.39,2400.73,11992.17


wariant,n_segments,rate_weighted_kg_h,median_rate_kg_h,p99_rate_kg_h
Wyłącznie 5–60 min,82674,3869.25,2397.22,10524.01


## 3. Porównanie typów samolotów — liczba lotów i tempo ważone czasem

W EDA najwięcej przedziałów miały **A20N (40 488)**, **A320 (31 980)** i **A359 (23 942)**. Ich surowe średnie `fuel_kg` nie są porównywalną miarą efektywności. Dla każdego typu raportujemy liczbę przedziałów i **unikalnych lotów**, sumę paliwa, liczbę godzin, tempo ważone czasem i medianę temp.

Próg `MIN_FLIGHTS = 30` ma charakter prezentacyjny i można go zmienić. Szczególnie ostrożnie interpretuj B38M (20 lotów) i B39M (1 lot w EDA).

In [0]:
MIN_FLIGHTS = 30

def type_summary(df):
    return (df.groupBy(aircraft_col)
        .agg(F.count("*").alias("n_segments"),
             F.countDistinct("flight_id").alias("n_flights"),
             F.sum("fuel_kg").alias("sum_fuel_kg"),
             F.sum("dur_s").alias("sum_duration_s"),
             F.expr("percentile_approx(rate_kg_h, 0.5)").alias("median_segment_rate_kg_h"),
             F.expr("percentile_approx(rate_kg_h, 0.95)").alias("p95_segment_rate_kg_h"))
        .withColumn("hours_observed", F.col("sum_duration_s") / 3600)
        .withColumn("weighted_rate_kg_h", F.col("sum_fuel_kg") / F.col("hours_observed"))
        .withColumn("weighted_co2_kg_h", F.col("weighted_rate_kg_h") * 3.16))

by_type_all = type_summary(segments)
by_type = by_type_all.filter(F.col("n_flights") >= MIN_FLIGHTS)

display(by_type.orderBy("weighted_rate_kg_h"))
print("Typy poniżej progu liczby lotów (nie usuwamy ich z danych):")
display(by_type_all.filter(F.col("n_flights") < MIN_FLIGHTS).orderBy("n_flights"))

flight_aircraft_type,n_segments,n_flights,sum_fuel_kg,sum_duration_s,median_segment_rate_kg_h,p95_segment_rate_kg_h,hours_observed,weighted_rate_kg_h,weighted_co2_kg_h
A20N,25789,3169,4802004.791165622,1.3574611719999967E7,1077.9103076300478,2394.82717330566,3770.7254777777684,1273.4962593977077,4024.248179696757
A21N,1426,361,439266.98555999977,803514.155,2105.122121012328,3426.613874245803,223.1983763888889,1968.0563661208919,6219.058116942018
B738,5737,532,1873775.7990399979,2880639.618999997,2393.1230552474594,4122.974561715332,800.1776719444437,2341.699680880491,7399.7709915823525
A321,1157,274,534046.350148,802950.517,2385.0187265917602,4896.002878359335,223.04181027777778,2394.377760308236,7566.233722574026
A320,17405,2971,6976480.556150399,1.0418867611999981E7,2395.4805267395514,4782.972617481765,2894.1298922222168,2410.5623506737656,7617.3770281291
B788,7047,583,6463614.503200002,5100100.463999997,4762.889155331377,5956.49771171213,1416.6945733333325,4562.461538898819,14417.378462920267
A333,905,146,1107266.2308559997,735442.9629999996,5400.094501653779,9319.285521443357,204.28971194444435,5420.078281558866,17127.44736972602
B789,1430,215,2613447.701487998,1723086.3880000003,5491.611346408275,7057.740728307537,478.6351077777779,5460.2089546289135,17254.260296627366
A359,15005,1532,2.681842375521598E7,1.6853428817999985E7,5645.360111650986,7251.815583844588,4681.508004999996,5728.586542322061,18102.333473737715
A332,3222,516,3677785.231326401,2282148.1969999974,5258.519391864036,11509.420728375388,633.9300547222215,5801.56312818762,18332.939485072882


Typy poniżej progu liczby lotów (nie usuwamy ich z danych):


flight_aircraft_type,n_segments,n_flights,sum_fuel_kg,sum_duration_s,median_segment_rate_kg_h,p95_segment_rate_kg_h,hours_observed,weighted_rate_kg_h,weighted_co2_kg_h
MD11,3,1,7200.0,2815.934,9360.948130317769,12533.357007452125,0.7822038888888889,9204.76119113587,29087.045363989353
A318,4,1,1267.336048,5379.3189999999995,593.7090576125859,982.6411746754845,1.4942552777777776,848.1389136431583,2680.1189671123802
B39M,1,1,1436.979456,1317.365,3926.873752984177,3926.873752984177,0.36593472222222223,3926.873752984177,12408.92105943
B763,8,2,4082.3279999999995,4036.8959999999997,3692.0616952243854,4465.378956910129,1.12136,3640.5150888207177,11504.02768067347
B77L,15,2,103400.0,28281.983,13525.303588797375,15635.450778732315,7.856106388888889,13161.736219132867,41591.08645245986
A388,6,3,25080.3456,5210.778,15819.109735657656,28054.597382835036,1.4474383333333334,17327.401812167012,54754.58972644776
B748,14,3,85536.496,17008.674,13490.546821832391,49532.604837134044,4.724631666666666,18104.37342734654,57209.820030415074
B752,14,3,6767.592640000001,8787.587,2537.9008032189195,5608.69401373896,2.440996388888889,2772.4713853757585,8761.009577787398
B38M,4,4,1400.0,2049.2380000000003,1971.3416211820656,3790.7665351130277,0.5692327777777778,2459.4507812172133,7771.864468646394
A306,44,9,101201.0368,41403.558,8062.326261068202,18888.81601537031,11.500988333333332,8799.333923910599,27805.895199557493


## 4. Porównanie A20N i A320 w zbliżonych warunkach

Te typy mają względnie duże próby. Ograniczamy porównanie do **wspólnych przedziałów długości trwania i dystansu lotu**. W każdej warstwie liczymy osobne tempo paliwa na godzinę, a następnie ważymy obie grupy jednakowymi wagami opartymi na mniejszej liczbie przedziałów w warstwie. Wymagamy co najmniej 30 przedziałów i 5 różnych lotów z każdego typu w warstwie.

To **porównanie opisowe**, nie dowód przewagi technologicznej ani przyczynowego wpływu generacji. Nie kontroluje masy samolotu, obciążenia, dokładnej fazy lotu, pogody ani warunków silników.

In [0]:
pair_types = ["A20N", "A320"]
matched_source = (segments
    .filter(F.col(aircraft_col).isin(pair_types))
    .filter(F.col("route_km").isNotNull() & (F.col("route_km") >= 0))
    .withColumn("duration_bin", F.when(F.col("dur_s") < 600, "05–10 min")
         .when(F.col("dur_s") < 1200, "10–20 min")
         .when(F.col("dur_s") < 2400, "20–40 min").otherwise("40–60 min"))
    .withColumn("route_bin", F.when(F.col("route_km") < 1000, "<1000 km")
         .when(F.col("route_km") < 3000, "1000–2999 km")
         .when(F.col("route_km") < 6000, "3000–5999 km").otherwise("6000+ km"))
)
strata = (matched_source.groupBy("duration_bin", "route_bin", aircraft_col)
    .agg(F.count("*").alias("n_segments"),
         F.countDistinct("flight_id").alias("n_flights"),
         F.sum("fuel_kg").alias("fuel_kg"),
         F.sum("dur_s").alias("dur_s"))
    .withColumn("rate_kg_h", F.col("fuel_kg") / F.col("dur_s") * 3600))

w = Window.partitionBy("duration_bin", "route_bin")
common = (strata
    .withColumn("types_present", F.count("*").over(w))
    .withColumn("min_segments", F.min("n_segments").over(w))
    .withColumn("min_flights", F.min("n_flights").over(w))
    .filter((F.col("types_present") == 2) & (F.col("min_segments") >= 30) & (F.col("min_flights") >= 5))
    .withColumn("shared_weight", F.col("min_segments")))

display(common.select("duration_bin", "route_bin", aircraft_col, "n_segments", "n_flights", "rate_kg_h", "shared_weight")
        .orderBy("duration_bin", "route_bin", aircraft_col))

matched_result = (common.groupBy(aircraft_col)
    .agg(F.sum(F.col("rate_kg_h") * F.col("shared_weight")).alias("weighted_sum"),
         F.sum("shared_weight").alias("total_weight"),
         F.count("*").alias("n_common_strata"))
    .withColumn("matched_rate_kg_h", F.col("weighted_sum") / F.col("total_weight")))

print("Porównanie A20N/A320 w warstwach wspólnych:")
display(matched_result.select(aircraft_col, "n_common_strata", "matched_rate_kg_h")
        .orderBy(aircraft_col))

if matched_result.count() != 2:
    print("Brak dostatecznej liczby wspólnych warstw — nie wyciągaj wniosku porównawczego.")

duration_bin,route_bin,flight_aircraft_type,n_segments,n_flights,rate_kg_h,shared_weight
05–10 min,1000–2999 km,A20N,6495,1159,1264.22726454912,6495
05–10 min,1000–2999 km,A320,9348,1889,2376.96945129502,6495
05–10 min,3000–5999 km,A20N,13863,1791,1197.4183049970743,2535
05–10 min,3000–5999 km,A320,2535,615,2670.0302409980854,2535
05–10 min,<1000 km,A20N,110,36,1425.8628494450677,110
05–10 min,<1000 km,A320,245,96,2631.0297646847375,110
10–20 min,1000–2999 km,A20N,1540,789,1732.7373109131202,1540
10–20 min,1000–2999 km,A320,2776,1348,2370.022337800711,1540
10–20 min,3000–5999 km,A20N,1546,1011,1158.5600475646693,502
10–20 min,3000–5999 km,A320,502,277,2418.9344138283923,502


Porównanie A20N/A320 w warstwach wspólnych:


flight_aircraft_type,n_common_strata,matched_rate_kg_h
A20N,10,1308.9093524677874
A320,10,2442.907539431919


## 5. Fazy lotu jako przybliżenie i analiza braków

`vr_mean` ma około **35% braków**. Znak średniej prędkości pionowej daje tylko **uproszczoną etykietę** (wznoszenie/zniżanie), a nie wiarygodne rozpoznanie fazy całego przedziału. Kategoria `blisko zera` wymaga znajomości jednostek i progu — tutaj oznaczamy dokładnie zero jako osobną kategorię, bez arbitralnych progów. Raportujemy również `brak danych`, aby nie zawyżać reprezentatywności porównania.

In [0]:
if "vr_mean" in segments.columns:
    phase_df = segments.withColumn(
        "vertical_motion_proxy",
        F.when(F.col("vr_mean").isNull(), "brak danych")
         .when(F.col("vr_mean") > 0, "średnie wznoszenie")
         .when(F.col("vr_mean") < 0, "średnie zniżanie")
         .otherwise("średnio 0")
    )
    phase_result = (phase_df.groupBy(aircraft_col, "vertical_motion_proxy")
        .agg(F.count("*").alias("n_segments"),
             F.countDistinct("flight_id").alias("n_flights"),
             F.sum("fuel_kg").alias("fuel_kg"),
             F.sum("dur_s").alias("dur_s"))
        .withColumn("rate_kg_h", F.col("fuel_kg") / F.col("dur_s") * 3600))
    display(phase_result.filter(F.col(aircraft_col).isin("A20N", "A320", "B738"))
            .orderBy(aircraft_col, "vertical_motion_proxy"))
else:
    print("Brak vr_mean — pomijam uproszczoną analizę ruchu pionowego")

flight_aircraft_type,vertical_motion_proxy,n_segments,n_flights,fuel_kg,dur_s,rate_kg_h
A20N,brak danych,4020,972,490918.06919200026,1571813.8659999978,1124.37298545323
A20N,średnie wznoszenie,10394,2495,2315631.277488005,5815146.921999992,1433.5446224787283
A20N,średnie zniżanie,10995,2616,1940996.3528216,6022483.298999994,1160.250103361518
A20N,średnio 0,380,326,54459.091663999876,165167.6330000001,1186.992429626932
A320,brak danych,1362,531,350979.5644640001,675141.7260000003,1871.498062423711
A320,średnie wznoszenie,7563,2388,3716622.9234783994,4641764.147999991,2882.4908155420276
A320,średnie zniżanie,8118,2547,2825509.115599999,4970435.785999995,2046.4670009037327
A320,średnio 0,362,305,83368.95260799999,131525.95200000002,2281.893609778243
B738,brak danych,286,92,78351.11279999999,171441.532,1645.24898249276
B738,średnie wznoszenie,2328,478,886592.6811200003,1164221.3619999995,2741.517855804472


## 6. Analiza wrażliwości: krótkie przedziały i skrajne tempa

Porównujemy dla poszczególnych typów trzy warianty: wszystkie dodatnie przedziały; wyłącznie 5–60 minut; 5–60 minut po odcięciu górnego 1% indywidualnego tempa **w całym zbiorze treningowym**. Ten ostatni wariant jest wyłącznie diagnostyczny — **nie oznacza, że obserwacje skrajne są błędne**. Uśrednienie po typie może zmieniać się po odcięciu innego składu próby.

In [0]:
p99 = segments.approxQuantile("rate_kg_h", [0.99], 0.001)[0]
variants = [
    ("wszystkie >0 s", base),
    ("5–60 min", segments),
    ("5–60 min, bez globalnego top 1% rate", segments.filter(F.col("rate_kg_h") <= F.lit(p99)))
]
compare = None
for label, df in variants:
    t = (type_summary(df)
         .filter(F.col(aircraft_col).isin("A20N", "A320", "A359", "B738", "B788"))
         .select(aircraft_col, "n_segments", "n_flights", "weighted_rate_kg_h")
         .withColumn("wariant", F.lit(label)))
    compare = t if compare is None else compare.unionByName(t)
print("Próg p99 indywidualnego tempa w przedziałach 5–60 min:", round(p99, 2), "kg/h")
display(compare.orderBy(aircraft_col, "wariant"))

Próg p99 indywidualnego tempa w przedziałach 5–60 min: 10757.9 kg/h


flight_aircraft_type,n_segments,n_flights,weighted_rate_kg_h,wariant
A20N,25789,3169,1273.4962593977007,5–60 min
A20N,25789,3169,1273.4962593977007,"5–60 min, bez globalnego top 1% rate"
A20N,40488,3220,1270.7041629079001,wszystkie >0 s
A320,17405,2971,2410.5623506737593,5–60 min
A320,17396,2971,2407.167948677901,"5–60 min, bez globalnego top 1% rate"
A320,31980,3023,2450.807632731345,wszystkie >0 s
A359,15005,1532,5728.586542322058,5–60 min
A359,14844,1520,5661.246700140146,"5–60 min, bez globalnego top 1% rate"
A359,23942,1571,5784.165176625652,wszystkie >0 s
B738,5737,532,2341.6996808804897,5–60 min


## 7. Analiza poziomu lotu — wyłącznie zaobserwowane odcinki

Suma `fuel_kg` po `flight_id` obejmuje **jedynie dostępne przedziały**, niekoniecznie cały lot. Nie oznaczamy tej sumy jako pełnego zużycia paliwa ani nie dzielimy jej przez `route_km` jako paliwo na kilometr całego lotu.

In [0]:
by_flight = (segments.groupBy("flight_id", aircraft_col)
    .agg(F.count("*").alias("observed_segments"),
         F.sum("fuel_kg").alias("observed_fuel_kg"),
         F.sum("dur_s").alias("observed_duration_s"))
    .withColumn("observed_rate_kg_h", F.col("observed_fuel_kg") / F.col("observed_duration_s") * 3600))
display(by_flight.orderBy(F.desc("observed_fuel_kg")).limit(20))

flight_id,flight_aircraft_type,observed_segments,observed_fuel_kg,observed_duration_s,observed_rate_kg_h
prc790175280,B744,7,78471.416,6001.333,47072.39168364762
prc789730137,A359,12,69600.0,20750.702999999998,12074.771635447727
prc772645763,A359,34,64410.064000000006,40529.79000000001,5721.130812668904
prc772643878,A359,29,60418.454399999995,36029.198,6036.94913886232
prc778812243,B77L,7,59300.0,16077.961000000001,13277.803074656045
prc772167552,A359,36,58700.0,38443.80999999999,5496.853719753585
prc783300782,A333,28,57480.0,33301.323000000004,6213.807181174153
prc779381999,B77W,16,57199.999999999985,25790.788,7984.24615796927
prc779574304,A359,11,55700.0,15324.413000000002,13085.003647448028
prc772635104,B77W,7,55557.0,13797.322999999999,14495.94243752937



## 8. Wnioski z analizy efektywności paliwowej

### 8.1. Różnice w zużyciu paliwa między typami samolotów

Analiza wykazała wyraźne różnice w obserwowanym tempie zużycia paliwa pomiędzy typami samolotów.

Wśród analizowanych typów:

- **A20N:** 1273,50 kg/h
- **A320:** 2410,56 kg/h
- **B738:** 2341,70 kg/h
- **B788:** 4562,46 kg/h
- **A359:** 5728,59 kg/h
- **B744:** 10 434,97 kg/h

Najniższe tempo zużycia paliwa spośród wymienionych typów zaobserwowano dla A20N, a najwyższe dla B744.

Różnice wynikają prawdopodobnie zarówno z charakterystyki konstrukcyjnej samolotów, jak i z ich wielkości, przeznaczenia oraz warunków wykonywanych lotów.

Nie oznacza to jednak, że samolot zużywający mniej paliwa na godzinę jest automatycznie bardziej efektywny w przeliczeniu na pasażera lub przewieziony ładunek.

### 8.2. Porównanie A20N i A320

Szczegółowa analiza A20N i A320 wykazała istotną różnicę w obserwowanym tempie spalania.

Po uwzględnieniu wspólnych przedziałów czasu trwania i dystansu trasy uzyskano:

| Typ samolotu | Tempo zużycia paliwa |
|---|---:|
| A20N | 1308,91 kg/h |
| A320 | 2442,91 kg/h |

W porównywanych warstwach A20N wykazuje około **46,4% niższe obserwowane tempo zużycia paliwa** niż A320.

Wynik sugeruje korzystniejszą charakterystykę zużycia paliwa A20N w analizowanej próbie.

Nie można jednak przypisać całej różnicy generacji samolotu, ponieważ analiza nie uwzględnia m.in. masy, obciążenia, pogody i wszystkich warunków operacyjnych.

### 8.3. Wpływ wznoszenia i zniżania

Zaobserwowano wyższe tempo zużycia paliwa podczas przedziałów zaklasyfikowanych jako średnie wznoszenie niż podczas średniego zniżania.

| Typ | Wznoszenie [kg/h] | Zniżanie [kg/h] |
|---|---:|---:|
| A20N | 1433,54 | 1160,25 |
| A320 | 2882,49 | 2046,47 |
| B738 | 2741,52 | 2098,35 |

Wyniki są zgodne z oczekiwaniem, że wznoszenie wymaga większego zapotrzebowania na energię niż zniżanie.

Podział oparto jednak na znaku średniej prędkości pionowej (`vr_mean`), dlatego stanowi on jedynie przybliżenie fazy lotu.

### 8.4. Stabilność otrzymanych wyników

Analiza wrażliwości wykazała, że główne różnice pomiędzy typami samolotów utrzymują się po:

- Ograniczeniu analizy do przedziałów trwających 5–60 minut.
- Pominięciu obserwacji o najwyższych wartościach indywidualnego tempa zużycia paliwa.

Przykładowo dla A320 tempo zużycia paliwa wynosiło:

- Wszystkie dodatnie przedziały: **2450,81 kg/h**.
- Przedziały 5–60 minut: **2410,56 kg/h**.
- Po dodatkowym ograniczeniu wartości skrajnych: **2407,17 kg/h**.

Oznacza to, że obserwowane różnice nie wynikają wyłącznie z krótkich przedziałów ani pojedynczych ekstremalnych pomiarów.

### 8.5. Ograniczenia analizy

Uzyskane wyniki należy interpretować jako porównanie **obserwowanego tempa zużycia paliwa**, a nie pełnej efektywności transportowej samolotów.

Analiza nie uwzględnia liczby pasażerów, masy przewożonego ładunku ani pełnego zestawu warunków operacyjnych.

Ponadto dostępne przedziały nie muszą obejmować całego lotu, dlatego suma ich zużycia paliwa nie zawsze odpowiada rzeczywistemu zużyciu podczas całej podróży.

### 8.6. Wniosek końcowy

Analiza wykazała, że obserwowane tempo zużycia paliwa różni się znacząco pomiędzy typami samolotów oraz pomiędzy przedziałami o różnej charakterystyce ruchu pionowego.

**Najważniejszym wynikiem jest niższe tempo zużycia paliwa A20N w porównaniu z A320**, utrzymujące się również po częściowym uwzględnieniu czasu trwania przedziałów i dystansu trasy.

Uzyskane rezultaty stanowią podstawę do kolejnego etapu projektu, w którym porównamy starsze i nowsze generacje samolotów oraz sprawdzimy, czy podobne zależności występują również w innych rodzinach konstrukcyjnych.
